# Viagens a serviço do governo federal em 2023

Este notebook explora como a quantidade, a duração e as despesas de viagens se distribuem entre cargos públicos. A análise usa dados do Portal da Transparência e apresenta apenas resumos agregados, sem linhas individuais.

**Perguntas:** quais cargos concentram mais viagens? Como variam a quantidade de viagens e a mediana das despesas? Como a mediana das despesas muda entre faixas de duração?

Para executar, coloque `2023_Viagem.csv` em `data/raw/`. Os dados não estão incluídos neste repositório.


## 1. Carregar e preparar os dados

O arquivo original contém informações de identificação e detalhes de itinerário. Primeiro calculamos a duração usando as datas; em seguida, removemos os campos pessoais e os detalhes que não são necessários para as perguntas desta análise.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from analise_viagens import (
    carregar_viagens,
    resumir_qualidade,
    resumir_por_cargo,
    resumir_por_duracao,
    criar_graficos,
)

arquivo = Path("data/raw/2023_Viagem.csv")
if not arquivo.exists():
    raise FileNotFoundError("Coloque 2023_Viagem.csv na pasta data/raw/. Consulte o README.md.")

df = carregar_viagens(arquivo)


In [ ]:
resumo_qualidade = resumir_qualidade(df)
resumo_qualidade


## 2. Comparar os cargos

Cerca de 39% dos registros não informam o cargo. A tabela de qualidade mostra a quantidade exata. Para que “não informado” não apareça como profissão, a comparação usa somente registros com cargo identificado e divulgado. O gráfico mostra os dez cargos mais frequentes entre os que representam pelo menos 1% desse subconjunto.

A despesa considerada é a soma de diárias, passagens e outros gastos, calculada apenas quando os três valores estão disponíveis. Devoluções não são descontadas. Cargos protegidos por sigilo também ficam fora da comparação.


In [ ]:
cargos_frequentes = resumir_por_cargo(df)
cargos_frequentes.round(2)


## 3. Despesas por duração

Agrupamos as viagens em faixas de duração e mostramos a mediana das despesas apenas nas faixas com pelo menos 30 registros e valores completos. A mediana reduz a influência de valores muito altos em comparação com a média.


In [ ]:
por_duracao = resumir_por_duracao(df)
por_duracao.round(2)


In [ ]:
total_viagens = int(resumo_qualidade.loc[
    resumo_qualidade["Indicador"] == "Viagens no arquivo", "Quantidade"
].iloc[0])
viagens_sem_cargo = int(resumo_qualidade.loc[
    resumo_qualidade["Indicador"] == "Viagens sem cargo informado", "Quantidade"
].iloc[0])
percentual_sem_cargo = viagens_sem_cargo / total_viagens * 100 if total_viagens else 0
fig, _ = criar_graficos(cargos_frequentes, por_duracao, percentual_sem_cargo)
plt.show()


## Limites da análise

Os dados são registros administrativos e podem conter valores ausentes ou inconsistências. A duração é a diferença entre as datas de início e fim, em dias; os gastos não descontam devoluções. O total por cargo pode incluir apenas as despesas com os três componentes preenchidos; a tabela de qualidade mostra quantos registros ficam sem esses valores completos. Comparações entre grupos descrevem padrões e não demonstram causa e efeito.

**Fonte:** [Portal da Transparência — viagens a serviço](https://portaldatransparencia.gov.br/download-de-dados/viagens). O arquivo de dados deve ser obtido diretamente da fonte.
